# Process and View Results From Multiple Longstrips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Angle,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),File size (MB),Total size (GB),ProcessingNotes,Unnamed: 23
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,2025-03-27,GHL_pyapp_20250327T1450,oven aging test day 2,strip 14,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1.0,2025-04-14,GHL_pyapp_20250414T1541,0.3-0.324 mg/mm,0.302 02-13 10,0.302,1-sided magnetic,7.0,9.0,1.50000,...,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.000000,8.100000,NaN,Accidentually deleted
7,2.0,2025-04-14,GHL_pyapp_20250414T1610,0.3-0.324 mg/mm,0.308 02-26 11,0.308,1-sided magnetic,7.0,9.0,1.50000,...,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.000000,8.100000,NaN,NaN
8,3.0,2025-04-14,GHL_pyapp_20250414T1618,0.3-0.324 mg/mm,0.302 02-13 10,0.302,1-sided magnetic,7.0,9.0,1.50000,...,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.000000,8.100000,NaN,Repeat test 1
9,4.0,2025-04-14,GHL_pyapp_20250414T1636,0.3-0.324 mg/mm,ES0331 #26 0.320,0.320,1-sided magnetic,7.0,9.0,1.50000,...,NaN,104.8,-60.5,165.3,"Medium, 48kHz",2.0,425.000000,8.100000,NaN,NaN


In [5]:
#%% Filter The Tests To View
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     (dftests['Test date']=='2025-05-05')
# ]
dfmasks = [
    (dftests['Test date']=='2025-05-08')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Angle,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),File size (MB),Total size (GB),ProcessingNotes,Unnamed: 23
22,17.0,2025-05-08,GHL_pyapp_20250508T1112,0.325-0.35,ES0408#17 0.327,0.327,Vacuum,9.0,3.5,2.39775,...,0.0,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
23,18.0,2025-05-08,GHL_pyapp_20250508T1128,0.325-0.35,ES0409#8 0.333,0.333,Vacuum,9.0,3.5,2.39775,...,0.0,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218077,4.143459,"done,5",NUM_FOVS: remove +1
24,19.0,2025-05-08,GHL_pyapp_20250508T1134,0.325-0.35,ES0410#18 0.333,0.333,Vacuum,9.0,3.5,2.39775,...,0.0,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218080,4.143517,"done,5",NUM_FOVS: remove +1
25,20.0,2025-05-08,GHL_pyapp_20250508T1143,0.375-0.4,ES0401#21 0.391,0.391,Vacuum,9.0,3.5,2.39775,...,0.0,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218082,4.143558,"done,5",NUM_FOVS: remove +1
26,21.0,2025-05-08,GHL_pyapp_20250508T1151,0.375-0.4,ES0403#33 0.392,0.392,Vacuum,9.0,3.5,2.39775,...,0.0,308.0,143.0,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218082,4.143550,"done,5",NUM_FOVS: remove +1
27,22.0,2025-05-08,GHL_pyapp_20250508T1158,0.375-0.4,ES0410#24 0.380,0.380,Vacuum,9.0,3.5,2.39775,...,0.0,308.0,143.0,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,0.218091,4.143722,"done,5",NUM_FOVS: remove +1
28,23.0,2025-05-08,GHL_pyapp_20250508T1539,0.325-0.349mg/mm lrg bag,ES0410#20 0.338,0.338,Vacuum,9.0,3.5,2.39775,...,0.0,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,0.218090,4.143710,"done,5",Simon (my data fields from today to be populat...
29,24.0,2025-05-08,GHL_pyapp_20250508T1545,0.325-0.349mg/mm lrg bag,ES0408#3 0.334,0.334,Vacuum,9.0,3.5,2.39775,...,0.0,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,0.218089,4.143700,"done,5",Simon
30,25.0,2025-05-08,GHL_pyapp_20250508T1551,0.325-0.349mg/mm lrg bag,ES0408#13 0.341,0.341,Vacuum,9.0,3.5,2.39775,...,0.0,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,0.218086,4.143628,"done,5",Simon
31,26.0,2025-05-08,GHL_pyapp_20250508T1555,0.325-0.349mg/mm lrg bag,ES0408#30 0.349,0.349,Vacuum,9.0,3.5,2.39775,...,0.0,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,0.218091,4.143720,"done,5",Simon


In [6]:
#%% Load OCT study information
octstudies = [];
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


STUDY: GHL_pyapp_20250508T1112
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1128
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1134
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1143
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1151
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

# Parse Through The OCTSTUDIES

In [7]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    pp.pprint(octstudy.resultsCheck())
    doWeRunTheNotebook = any([v is False for k,v in octstudy.resultsCheck().items()])
    print('Run?',doWeRunTheNotebook)
    if(not doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

    print('~~~~~~~');
    print('We will analyze data for {:} octstudies.'.format(len(octstudies_to_run)))
    print([x.name for x in octstudies_to_run]);

~~~~~~~
GHL_pyapp_20250508T1112
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD',
                                      '/dfstepE'],
    'exist_data_extracted': True,
    'figoutmp4_mazetest': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
We will analyze data for 1 octstudies.
['GHL_pyapp_20250508T1112']
~~~~~~~
GHL_pyapp_20250508T1128
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD',
                                      '/dfstepE'],
    'exist_data_extracted': True,
    'figoutmp4_mazetest': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
We will analyze data for 2 octstudies.
['GHL_pyapp_20250508T1112', 'GHL_pyapp_202

# 1. From stored dataframes load summary data we care about

In [8]:
#octstudies_to_run = [octstudies_to_run[0]]
dfs = [];
for idx,octstudy in enumerate(octstudies_to_run):
    print(octstudy.name)

    octstudy.load_previously_saved_merged_volume();

    data_extracted = octstudy.load_data_extracted();
    dfstep = octstudy.load_data_extracted_along_strip();
    
    dfstep['octstudy'] = octstudy;

    dfs.append(dfstep)
    #data_extracted = 


    # Calculate Path From Top To Bottom
dfsteps = pd.concat(dfs,keys=[x.name for x in octstudies_to_run]);

GHL_pyapp_20250508T1112
Loading GHL_pyapp_20250508T1112_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250508T1128
Loading GHL_pyapp_20250508T1128_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250508T1134
Loading GHL_pyapp_20250508T1134_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250508T1143
Loading GHL_pyapp_20250508T1143_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250508T1151
Loading GHL_pyapp_20250508T1151_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250508T1158
Loading GHL_pyapp_20250508T1158_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
L

In [9]:
dfsteps

pixel_depth_strip_top_sum_threshold  \
                        slice                                        
GHL_pyapp_20250508T1112 5                                    57.76   
                        10                                   56.80   
                        15                                   59.52   
                        20                                   56.60   
                        25                                   58.08   
...                                                            ...   
GHL_pyapp_20250508T1620 8250                                 53.52   
                        8255                                 53.92   
                        8260                                 53.64   
                        8265                                 53.28   
                        8270                                 54.10   

                               pixel_depth_strip_top  pixel_depth_strip_bot  \
                        slice                                                 
GHL_pyapp_20250508T1112 5                        181                    247   
                        10                       182                    246   
                        15                       183                    244   
                        20                       184                    244   
                        25                       184                    243   
...                                              ...                    ...   
GHL_pyapp_20250508T1620 8250                     268                    331   
                        8255                     266                    329   
                        8260                     263                    329   
                        8265                     260                    327   
                        8270                     257                    322   

                               pixel_depth_wax_center  \
                        slice                           
GHL_pyapp_20250508T1112 5                         222   
                        10                        223   
                        15                        226   
                        20                        231   
                        25                        223   
...                                               ...   
GHL_pyapp_20250508T1620 8250                      313   
                        8255                      300   
                        8260                      302   
                        8265                      303   
                        8270                      303   

                                               px_wax_transverse_edges  \
                        slice                                            
GHL_pyapp_20250508T1112 5      (56.33712121212121, 105.00769230769231)   
                        10     (56.755319148936174, 105.3936170212766)   
                        15      (56.82631578947368, 103.9396551724138)   
                        20     (55.68604651162791, 102.38636363636364)   
                        25     (53.572463768115945, 102.2843137254902)   
...                                                                ...   
GHL_pyapp_20250508T1620 8250   (75.21428571428571, 112.86585365853658)   
                        8255    (75.59433962264151, 111.8030303030303)   
                        8260            (74.046875, 108.2872340425532)   
                        8265   (73.17785234899328, 105.88461538461539)   
                        8270   (73.25882352941177, 107.01470588235294)   

                              wax_top_seed_candidate_px  \
                        slice                             
GHL_pyapp_20250508T1112 5                     (190, 80)   
                        10                    (195, 80)   
                        15                    (202, 79)   
                        20                    (203, 78)   
                        25       

In [10]:
import gc
gc.collect()

40

# handle the source image

In [11]:
def getSitkImage(octstudy_with_loaded_vdvol):
    # Test getting SITK image
    # Get ITK image without requiring new memory
    import itk
    import vtk

    # function to go from itk image to simpleitk image
    def itkToSimpleITK(itk_image):
        new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
        new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
        new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
        new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
        return new_sitk_image;

    print('Get volume for',octstudy_with_loaded_vdvol.name)

    # create simpleitk 3dimage, directly from the previously-loaded vtk 3dimage
    simgmerged = itkToSimpleITK( itk.image_from_vtk_image(octstudy_with_loaded_vdvol.vdvol.dataset) );



    # itk_image
    #print(itk_image)
    slicer_1 = slice( data_extracted['auto_bounds']['topdown_edge_left_px3']  ,  data_extracted['auto_bounds']['topdown_edge_right_px3']  )
    #print(' slicer_dim_0',slicer_0)
    print(' slicer_dim_1',slicer_1)
    #simgmerged.GetSize()
    return simgmerged,{'slicer_1':slicer_1};

simgmerged,voldata = getSitkImage(octstudy);
slicer_1 = voldata['slicer_1'];

Get volume for GHL_pyapp_20250508T1620
 slicer_dim_1 slice(195, 8468, None)


In [12]:
print( simgmerged[:,slicer_1,:] )

Image (000001DBF30ACC20)
  RTTI typeinfo:   class itk::Image<unsigned char,3>
  Reference Count: 1
  Modified Time: 1705
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 1692
  UpdateMTime: 1704
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 8273, 175]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 8273, 175]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 8273, 175]
  Spacing: [0.003475, 0.02, 0.02]
  Origin: [0, 3.9, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
0.003475 0 0
0 0.02 0
0 0 0.02

  PointToIndexMatrix: 
287.77 0 0
0 50 0
0 0 50

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (000001E091BEEE20)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,unsigned cha

In [13]:
slab_thickness = 10; # px, this was used in the bulk analysis
slice_position_along_strip_px = 4000;
slab_slicer = slice(slice_position_along_strip_px-slab_thickness//2,slice_position_along_strip_px+slab_thickness//2);
simg_slab_img = (simgmerged[:,slicer_1,:])[:,slab_slicer,:]

# 3. Plot Of Values Along Strip

In [13]:
slice_centers = dfsteps.index;

In [14]:
simgmerged_summed = sitk.SumProjection(simgmerged[:,slicer_1,:],projectionDimension=0);


In [15]:
print(simgmerged_summed)

Image (000001DBF30ACEF0)
  RTTI typeinfo:   class itk::Image<double,3>
  Reference Count: 1
  Modified Time: 1771
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 1760
  UpdateMTime: 1770
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1, 8273, 175]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1, 8273, 175]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1, 8273, 175]
  Spacing: [2.39775, 0.02, 0.02]
  Origin: [7.46251e+06, 3.9, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
2.39775 0 0
0 0.02 0
0 0 0.02

  PointToIndexMatrix: 
0.417058 0 0
0 50 0
0 0 50

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (000001DBE91194B0)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,double>
      R

In [16]:
ndaimg = sitk.GetArrayFromImage(sitk.RescaleIntensity(simgmerged_summed,0,255)); # numpy array


In [17]:
ndaimg

array([[[63.85712218],
        [62.47529728],
        [64.93551966],
        ...,
        [60.90075495],
        [89.5397937 ],
        [59.68909543]],

       [[66.33374605],
        [68.6586562 ],
        [69.12404827],
        ...,
        [59.76700247],
        [56.22018186],
        [63.79561662]],

       [[65.99956584],
        [73.72466413],
        [64.4270737 ],
        ...,
        [69.1199479 ],
        [61.88279372],
        [65.44396562]],

       ...,

       [[51.98244881],
        [54.24175303],
        [53.68615281],
        ...,
        [93.67911786],
        [45.64532598],
        [15.17957211]],

       [[59.24010484],
        [51.31203821],
        [54.27660618],
        ...,
        [53.6451491 ],
        [32.01979434],
        [ 4.51450808]],

       [[68.36137933],
        [60.47636659],
        [54.18434784],
        ...,
        [54.11054117],
        [49.83385459],
        [ 2.4294696 ]]])

In [18]:
#ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:

ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:
pil_img.show()

In [20]:
studyname = dfsteps.index.get_level_values(0).unique().tolist()[1];
octstudy = dfsteps.loc[studyname].iloc[0]['octstudy']

simgmerged,voldata = getSitkImage(octstudy);
slicer_1 = voldata['slicer_1'];
df = dfsteps.loc[studyname,:]

Get volume for GHL_pyapp_20250508T1128
 slicer_dim_1 slice(195, 8468, None)


In [63]:
# Make Detailed Metrics Plots Of 1-Device
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = 5
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )

slice_centers = df.index;

myrow = 1;
if True:
    # make image
    legendgroup = 'image';

    # use previously created simpleitk merged image filter
    #ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
    ndaimg = sitk.GetArrayFromImage(sitk.RescaleIntensity(simgmerged_summed,0,255)); # numpy array
    print(ndaimg.shape)
    from PIL import Image
    import base64
    from io import BytesIO
    pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object
    pil_img = pil_img.convert("L"); # grayscale image convert, from float
    prefix = "data:image/png;base64,"
    with BytesIO() as stream:
        pil_img.save(stream, format="png")
        base64_string = prefix + base64.b64encode(stream.getvalue()).decode("utf-8")

    fig.add_trace(
        go.Image(
            source=base64_string,
            #cmap='jet',
        ),
        row=myrow,col=1,
    )

legendgroup = 'distances';
myrow+=1;
# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-wax_top
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness1',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);


# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-df.loc[slice_centers]['pixel_depth_strip_top']
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness2',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0]),
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_width_px',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_top,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_top',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)




legendgroup = 'areas';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']-df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_holes',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_filled',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_convex',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)


legendgroup = 'ratios';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areafilled_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_areafilled_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

legendgroup = 'counts';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['num_paths'],
        name='crossovers_px_paths',
        legendgroup=legendgroup,
    ),
    row=myrow,col=1,
)
fig.update_yaxes(row=myrow,type='log')

# setup legends per row
for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
    legend_name = f"legend{i}"
    fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
    fig.update_traces(row=i, legend=legend_name)

fig.update_traces(xaxis='x{:}'.format(nrows))
fig.update_yaxes(title='pixels',row=1);
fig.update_yaxes(title='pixels^2',row=2);
fig.update_yaxes(title='ratios',row=3);
fig.update_yaxes(title='counts',row=4);
fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);
fig.update_layout(hovermode='x unified',hoversubplots="axis",spikedistance=-1);
#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

fig.show(renderer='browser')
#fig

(175, 8273, 1)


In [ ]:
fig.update_yaxes(row=1,scaleanchor=None)

In [53]:
fig.layout

Layout({
    'hovermode': 'x unified',
    'hoversubplots': 'axis',
    'legend': {'y': 1.0, 'yanchor': 'top'},
    'legend2': {'y': 0.796, 'yanchor': 'top'},
    'legend3': {'y': 0.592, 'yanchor': 'top'},
    'legend4': {'y': 0.388, 'yanchor': 'top'},
    'legend5': {'y': 0.184, 'yanchor': 'top'},
    'showlegend': True,
    'spikedistance': -1,
    'template': '...',
    'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'matches': 'x5', 'showticklabels': False},
    'xaxis2': {'anchor': 'y2', 'domain': [0.0, 1.0], 'matches': 'x5', 'showticklabels': False},
    'xaxis3': {'anchor': 'y3', 'domain': [0.0, 1.0], 'matches': 'x5', 'showticklabels': False},
    'xaxis4': {'anchor': 'y4', 'domain': [0.0, 1.0], 'matches': 'x5', 'showticklabels': False},
    'xaxis5': {'anchor': 'y5', 'domain': [0.0, 1.0], 'title': {'text': 'distance along strip (px)'}},
    'yaxis': {'anchor': 'x', 'domain': [0.816, 1.0], 'title': {'text': 'pixels'}},
    'yaxis2': {'anchor': 'x2', 'domain': [0.6120000000000001,

In [62]:
fig.layout.yaxis5

layout.YAxis({
    'anchor': 'x5', 'domain': [0.0, 0.184], 'type': 'log'
})